# UNIT IV — Language Models

---

### Topics Covered
1. **Recurrent Neural Networks (RNN)**
2. **Long Short-Term Memory (LSTM)**
3. **Attention Mechanism & Transformers**
4. **Self-Attention & Multi-Head Attention**
5. **BERT** — Bidirectional Encoder Representations from Transformers
6. **RoBERTa** — Robustly Optimized BERT Pretraining Approach
7. **Fine-Tuning for Downstream Tasks**

---
# 1. Evolution of Neural Networks & LLMs

## 1.1 Early Neural Networks (1950s – 1980s)

- The idea of artificial neurons was introduced by **McCulloch & Pitts (1943)**, leading to the first generation of neural networks.
- Early **Feedforward Neural Networks (FNNs)** were developed, but they lacked memory and could not handle sequential data effectively.
- In the 1980s, researchers started exploring networks that could handle time-dependent data, leading to the development of basic **Recurrent Neural Networks (RNNs)**.

## 1.2 Introduction of Basic RNNs (1986 – 1990s)

- **1986**: David Rumelhart, Geoffrey Hinton, and Ronald Williams introduced **Backpropagation Through Time (BPTT)**.
- **1990**: Elman & Jordan Networks were developed, introducing simple RNN architectures with feedback loops.
- These models suffered from the **vanishing gradient problem**, making it difficult to learn long-term dependencies.

## 1.3 LSTM Networks (1997)

- **1997**: Sepp Hochreiter and Jürgen Schmidhuber introduced **Long Short-Term Memory (LSTM)** to solve the vanishing gradient problem.
- Key Feature: LSTM introduced **gates** (input, forget, and output) to regulate information flow.

## 1.4 GRU (2014)

- **2014**: Kyunghyun Cho et al. introduced the **Gated Recurrent Unit (GRU)**, a simplified LSTM with only two gates (update and reset).

## 1.5 Attention Mechanisms & Transformers (2014 – Present)

- **2014**: **Attention Mechanisms** introduced by Bahdanau et al.
- **2017**: Google introduced the **Transformer** model (*"Attention is All You Need"*), eliminating the need for RNNs.
- Transformers like **BERT**, **GPT**, and **T5** have largely replaced RNNs in NLP tasks.

---
# 2. Recurrent Neural Networks (RNNs)

## 2.1 What is Sequential Data?

If there is a particular order in which related things follow each other, we call it a **sequence**.

- *"i am a good boy"* and *"am i a good boy"* — same words, different meaning. **Position matters!**

## 2.2 Why RNNs?

Traditional **Feedforward Neural Networks (FNNs)** process each input independently — they cannot capture context or order. RNNs were introduced to handle:
- Language modeling
- Machine translation
- Speech recognition
- Time series analysis

## 2.3 How RNNs Work

RNNs process a sequence step-by-step, passing a **hidden state** from one time step to the next:

- At `t=0`: input `X₀ = "i"` → produces hidden state `h₀`
- At `t=1`: input `X₁ = "am"` + `h₀` → produces `h₁`
- And so on...

**Key formula:**
```
aₜ = f(U·Xₜ + W·aₜ₋₁ + b)
```
Where:
- `U` = input → hidden weight
- `W` = hidden → hidden (recurrent) weight
- `V` = hidden → output weight
- `b` = bias
- `f` = activation function

## 2.4 Applications of RNNs

- **NLP**: Language modeling, text generation
- **Speech Recognition**: Transcribing spoken words
- **Machine Translation**: Sequence-to-sequence translation
- **Time Series**: Stock market prediction, weather forecasting
- **Music Generation**: Predicting sequences of notes

In [ ]:
# Simple RNN forward pass demonstration
import numpy as np

np.random.seed(42)

# Hyperparameters
input_size = 3   # one-hot encoded chars: C, A, R
hidden_size = 2
output_size = 1

# Weight matrices (randomly initialized)
U = np.random.randn(hidden_size, input_size)   # input -> hidden
W = np.random.randn(hidden_size, hidden_size)  # hidden -> hidden
V = np.random.randn(output_size, hidden_size)  # hidden -> output
b_h = np.zeros((hidden_size, 1))
b_y = np.zeros((output_size, 1))

# Input sequence: C=0, A=1, R=2 (one-hot)
X = [
    np.array([[1], [0], [0]]),  # C
    np.array([[0], [1], [0]]),  # A
    np.array([[0], [0], [1]]),  # R
]

h = np.zeros((hidden_size, 1))  # initial hidden state

print("=== RNN Forward Pass: Predicting next character after 'CAR' ===")
for t, x in enumerate(X):
    h = np.tanh(U @ x + W @ h + b_h)
    y = V @ h + b_y
    print(f"t={t}: hidden={h.T.round(4)}, output={y.T.round(4)}")

print(f"\nFinal output (raw): {y.flatten()[0]:.4f}")
print("Note: With proper training, this would predict 'D'")

## 2.5 RNN Architectures

| Architecture | Description | Use Case |
|---|---|---|
| **One-to-One** | Single input → Single output | Image classification |
| **One-to-Many** | Single input → Sequence output | Image captioning |
| **Many-to-One** | Sequence input → Single output | Sentiment analysis |
| **Many-to-Many (Equal)** | Sequence → Same-length sequence | NER, video frame labeling |
| **Many-to-Many (Unequal)** | Sequence → Different-length sequence | Machine translation, Speech-to-Text |

---
# 3. Long Short-Term Memory (LSTM)

## 3.1 Problem with RNNs: Vanishing/Exploding Gradients

- **Vanishing Gradient**: Gradients shrink as they pass through many time steps → early information becomes irrelevant.
- **Exploding Gradient**: Gradients grow too large → unstable, erratic updates.

**LSTM** was designed by Hochreiter & Schmidhuber (1997) to solve this.

## 3.2 LSTM Architecture

LSTM introduces a **memory cell** controlled by **three gates**:

| Gate | Function |
|---|---|
| **Forget Gate** | Decides what information to remove from cell state |
| **Input Gate** | Decides what new information to add to cell state |
| **Output Gate** | Decides what to output from the cell state |

## 3.3 Working of Each Gate

### Forget Gate
```
fₜ = σ(Wf · [hₜ₋₁, xₜ] + bf)
```
- Output: 0 = forget, 1 = keep

### Input Gate
```
iₜ = σ(Wi · [hₜ₋₁, xₜ] + bi)
C̃ₜ = tanh(Wc · [hₜ₋₁, xₜ] + bc)
Cₜ = fₜ * Cₜ₋₁ + iₜ * C̃ₜ
```

### Output Gate
```
oₜ = σ(Wo · [hₜ₋₁, xₜ] + bo)
hₜ = oₜ * tanh(Cₜ)
```

## 3.4 Cell State — The "Conveyor Belt"

The **cell state** `Cₜ` is the long-term memory. It runs through the entire chain, with information added or removed by the gates. Think of it like a conveyor belt — information travels along it with only minor, regulated changes.

## 3.5 LSTM vs RNN

| Feature | RNN | LSTM |
|---|---|---|
| Memory | Short-term only | Short + Long term |
| Gates | None | Forget, Input, Output |
| Vanishing Gradient | Suffers | Solved |
| Complexity | Simple | More parameters |
| Long-range dependencies | Poor | Excellent |

In [ ]:
import torch
import torch.nn as nn

# Compare RNN vs LSTM on a simple sequence
input_size = 5
hidden_size = 10
seq_len = 20
batch_size = 1

rnn = nn.RNN(input_size, hidden_size, batch_first=True)
lstm = nn.LSTM(input_size, hidden_size, batch_first=True)

x = torch.randn(batch_size, seq_len, input_size)

rnn_out, _ = rnn(x)
lstm_out, (h_n, c_n) = lstm(x)

print("=== RNN vs LSTM Comparison ===")
print(f"Input shape:       {x.shape}")
print(f"RNN output shape:  {rnn_out.shape}")
print(f"LSTM output shape: {lstm_out.shape}")
print(f"LSTM hidden state: {h_n.shape}")
print(f"LSTM cell state:   {c_n.shape}")

rnn_params = sum(p.numel() for p in rnn.parameters())
lstm_params = sum(p.numel() for p in lstm.parameters())
print(f"\nRNN parameters:  {rnn_params}")
print(f"LSTM parameters: {lstm_params}  (~4x more due to gates)")

## 3.6 Applications of LSTM

- **Language Modeling**: Text generation, machine translation, summarization
- **Speech Recognition**: Speech-to-text transcription
- **Time Series Forecasting**: Stock prices, weather, energy consumption
- **Anomaly Detection**: Fraud detection, network intrusion detection
- **Recommender Systems**: Movie, music, and book recommendations
- **Video Analysis**: Activity recognition, action classification (combined with CNNs)

---
# 4. Attention Mechanism & Transformers

## 4.1 The Problem with RNNs for Long Sequences

RNNs and LSTMs process sequences **step-by-step** — information from early steps can get "washed out" in long sequences. They also cannot be parallelized during training.

**Attention** was introduced to let models focus on relevant parts of the input regardless of distance.

## 4.2 Text Generation with RNNs (Advanced Variants)

| Step | Description |
|---|---|
| Seed Sequence | Start with a few words |
| Predict & Append | Use RNN to predict the next word, append it |
| Repeat | Continue until desired length |

**Challenges**: Vanishing gradients, limited memory for long coherent text.

**Advanced Variants**: LSTM and GRU address these issues with gating mechanisms.

---
# 5. Self-Attention & Multi-Head Attention

## 5.1 Intuition — Self-Attention

> *"The animal didn't cross the street because it was too tired."*
> What does **"it"** refer to? The animal or the street?

You instinctively know it's **the animal** — because you paid more attention to that word. **Self-attention** lets each word look at every other word to build a richer, context-aware representation.

## 5.2 The 6 Components of Self-Attention

| # | Component | Description |
|---|---|---|
| 1 | Input Embeddings | Words → dense numerical vectors |
| 2 | Q, K, V Matrices | Query, Key, Value — learned projections |
| 3 | Score Computation | `Scores = Q · Kᵀ` — relevance between words |
| 4 | Scaling | Divide by `√d_k` to stabilize gradients |
| 5 | Softmax | Convert scores to probabilities (attention weights) |
| 6 | Weighted Sum | `Output = A · V` — context-enriched representation |

## 5.3 Q, K, V — Library Analogy

Think of it like a **library search system**:
- **Query (Q)** = The question you are asking: *"What am I looking for?"*
- **Key (K)** = The index/label of each book: *"What does each word offer?"*
- **Value (V)** = The actual book content: *"What information does each word contain?"*

**Math:**
```
Q = X · W_Q
K = X · W_K
V = X · W_V

Attention(Q, K, V) = softmax(Q·Kᵀ / √d_k) · V
```

## 5.4 Why Scale by √d_k?

If Q and K are random vectors with mean 0 and variance 1, their dot product has variance = `d_k`. Dividing by `√d_k` brings variance back to 1, stabilizing gradients and preventing the softmax from saturating.

In [ ]:
import torch
import torch.nn.functional as F
import math

# Self-Attention from scratch
def self_attention(X, W_Q, W_K, W_V):
    Q = X @ W_Q
    K = X @ W_K
    V = X @ W_V
    d_k = Q.shape[-1]
    scores = Q @ K.transpose(-2, -1) / math.sqrt(d_k)
    weights = F.softmax(scores, dim=-1)
    output = weights @ V
    return output, weights

torch.manual_seed(0)
n, d_model, d_k = 3, 4, 4  # 3 words: "I", "love", "cats"

# Input embeddings
X = torch.tensor([
    [0.1, 0.2, 0.3, 0.4],  # "I"
    [0.5, 0.6, 0.7, 0.8],  # "love"
    [0.9, 0.1, 0.2, 0.3],  # "cats"
])

W_Q = torch.randn(d_model, d_k)
W_K = torch.randn(d_model, d_k)
W_V = torch.randn(d_model, d_k)

output, attn_weights = self_attention(X, W_Q, W_K, W_V)

words = ["I", "love", "cats"]
print("=== Self-Attention: 'I love cats' ===")
print("\nAttention Weight Matrix (how much each word attends to others):")
print(f"{'':8}", "  ".join(f"{w:>6}" for w in words))
for i, row in enumerate(attn_weights):
    print(f"{words[i]:>6}  ", "  ".join(f"{v:.4f}" for v in row.tolist()))
print("\nOutput (context-enriched representations):")
for i, row in enumerate(output):
    print(f"  {words[i]}: {row.detach().numpy().round(4)}")

## 5.5 Multi-Head Attention

**Single-head** attention views relationships from ONE perspective. But understanding a sentence requires multiple perspectives simultaneously.

> *"The trophy didn't fit in the bag because it was too big."*
> - 🔍 Head 1 (Grammar): "it" is a pronoun → refers to "trophy"
> - 🔍 Head 2 (Size/Logic): "big" relates to size → "trophy"
> - 🔍 Head 3 (Position): "it" is close to "bag" — but context says trophy
> - 🔍 Head 4 (Meaning): "fit" and "big" are semantically linked

### Doctor Analogy
Like a patient seeing a **panel of specialists simultaneously**:
- 👨‍⚕️ Cardiologist → Heart
- 👩‍⚕️ Neurologist → Brain  
- 👨‍⚕️ Orthopedic → Bones
- 👩‍⚕️ General Physician → Overall health

Each examines the same patient from a different angle — then combine reports.

### Multi-Head Attention Steps
1. Input Embeddings (same as Self-Attention)
2. **Split** into `h` heads — each gets `d_k = d_model / h` dimensions
3. Each head computes its own **Q, K, V**
4. Each head runs **Self-Attention independently**
5. **Concatenate** all head outputs
6. **Final Linear Projection** via `W_O` matrix

```
MultiHead(Q,K,V) = Concat(head₁, ..., headₕ) · W_O
```

In [ ]:
import torch
import torch.nn as nn

# Multi-Head Attention using PyTorch built-in
d_model = 8
num_heads = 2
seq_len = 3  # "I", "love", "cats"

mha = nn.MultiheadAttention(embed_dim=d_model, num_heads=num_heads, batch_first=True)

torch.manual_seed(0)
X = torch.randn(1, seq_len, d_model)  # (batch, seq, embed)

output, attn_weights = mha(X, X, X)

words = ["I", "love", "cats"]
print("=== Multi-Head Attention (2 heads, d_model=8) ===")
print(f"Input shape:  {X.shape}")
print(f"Output shape: {output.shape}")
print(f"Attention weights shape: {attn_weights.shape}")
print("\nAverage attention weights across heads:")
print(f"{'':8}", "  ".join(f"{w:>6}" for w in words))
for i, row in enumerate(attn_weights[0]):
    print(f"{words[i]:>6}  ", "  ".join(f"{v:.4f}" for v in row.tolist()))
print(f"\nTotal MHA parameters: {sum(p.numel() for p in mha.parameters())}")

---
# 6. BERT — Bidirectional Encoder Representations from Transformers

## 6.1 What is BERT?

BERT is a large language model developed by **Google (2018)**. Key features:

- Uses **Transformer encoders only** (no decoder)
- Processes all words **in parallel** (unlike RNNs)
- **Bidirectional**: reads text left-to-right AND right-to-left simultaneously
- Generates **contextual embeddings** — the same word gets different embeddings depending on context

## 6.2 BERT Model Sizes

| Feature | BERT-Base | BERT-Large |
|---|---|---|
| Layers (Encoders) | 12 | 24 |
| Hidden Size | 768 | 1024 |
| Attention Heads | 12 | 16 |
| Parameters | ~110M | ~340M |

## 6.3 Input Representation

BERT's input embedding = **Token Embedding + Segment Embedding + Positional Encoding**

- **[CLS]** token: always the first token; used for classification tasks
- **[SEP]** token: separates two sentences
- **Segment encoding**: tells BERT which sentence (A or B) each token belongs to

```
[CLS] Sentence A tokens [SEP] Sentence B tokens [SEP]
```

## 6.4 BERT Pre-Training Tasks

### Masked Language Modeling (MLM)

15% of tokens are randomly masked. Of those:
- **80%** → replaced with `[MASK]` token
- **10%** → replaced with a random word
- **10%** → left unchanged

Example: *"my dog is hairy"* → *"my dog is [MASK]"*

BERT predicts the original word from context (bidirectionally).

### Next Sentence Prediction (NSP)

Two sentences are fed to BERT. The task: does Sentence B follow Sentence A?

- **50%** of the time: real consecutive pair → label = `IsNext`
- **50%** of the time: random pair → label = `NotNext`

The **[CLS] token output** is used to make this binary prediction.

## 6.5 Pre-Training Data

BERT was pre-trained on ~**3,300M words**:
- 800M words from **BooksCorpus**
- 2,500M words from **English Wikipedia**

In [ ]:
!pip install transformers -q

In [ ]:
from transformers import BertTokenizer, BertModel
import torch

tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertModel.from_pretrained('bert-base-uncased')
model.eval()

# Contextual embeddings demo
sentences = [
    "I went to the bank to deposit money.",
    "The river bank was covered in mud."
]

print("=== BERT Contextual Embeddings for the word 'bank' ===")
bank_embeddings = []
for sent in sentences:
    tokens = tokenizer(sent, return_tensors='pt')
    with torch.no_grad():
        output = model(**tokens)
    token_ids = tokens['input_ids'][0].tolist()
    token_list = tokenizer.convert_ids_to_tokens(token_ids)
    bank_idx = token_list.index('bank')
    bank_emb = output.last_hidden_state[0, bank_idx]
    bank_embeddings.append(bank_emb)
    print(f"\nSentence: {sent}")
    print(f"  'bank' embedding (first 5 dims): {bank_emb[:5].numpy().round(4)}")

cos_sim = torch.nn.functional.cosine_similarity(
    bank_embeddings[0].unsqueeze(0), bank_embeddings[1].unsqueeze(0)
).item()
print(f"\nCosine similarity between two 'bank' embeddings: {cos_sim:.4f}")
print("(Less than 1.0 = BERT gives different embeddings based on context!")

---
# 7. RoBERTa — Robustly Optimized BERT Pretraining Approach

## 7.1 Introduction

**RoBERTa** is a pretrained transformer model developed by **Meta AI**. It improves on BERT **without changing the transformer architecture** — only by optimizing the training procedure.

Key improvements:
- More training data
- Dynamic masking (vs BERT's static masking)
- Larger batch sizes and longer training
- Removal of Next Sentence Prediction (NSP)

## 7.2 Why RoBERTa? — BERT's Weaknesses

| Observation | Issue |
|---|---|
| Insufficient training data | BERT was undertrained |
| Limited iterations | More training helps |
| Static masking | Model sees same mask every epoch |
| NSP objective | Unnecessary; hurts performance |

## 7.3 Architecture of RoBERTa

RoBERTa uses the same **Transformer Encoder** as BERT:

1. **Input Embedding Layer** — Token embeddings + Positional embeddings
2. **Multi-Head Self-Attention** — Each word attends to every other word
3. **Feed Forward Neural Network** — Transforms attention output
4. **Residual Connections + Layer Normalization** — Stable learning
5. **Output Representation** — Contextual token embeddings

## 7.4 Dynamic Masking Strategy

RoBERTa **re-masks** the input at every epoch, so the model learns from different masked positions:

```
Original: "The dog runs in the garden"
Epoch 1:  "The dog [MASK] in the garden"  ← runs masked
Epoch 2:  "The [MASK] runs in the garden"  ← dog masked
Epoch 3:  "The dog runs in [MASK] garden"  ← the masked
```

## 7.5 Key Differences: BERT vs RoBERTa

| Feature | BERT | RoBERTa |
|---|---|---|
| Masking | Static (fixed at start) | Dynamic (changes each epoch) |
| NSP Task | Yes | Removed |
| Training Data | 3.3B words | 160GB+ (books, news, web) |
| Batch Size | 256 | 8,000 |
| Training Steps | 1M | More |
| Performance | Strong | Better on most benchmarks |

In [ ]:
from transformers import RobertaTokenizer, RobertaModel, pipeline
import torch

# Fill-mask demo with RoBERTa (dynamic masking)
fill_mask = pipeline('fill-mask', model='roberta-base')

sentences = [
    "The student completed the <mask>.",
    "The dog runs in the <mask>.",
    "Natural language <mask> is fascinating.",
]

print("=== RoBERTa Dynamic Masking Demo ===")
for sent in sentences:
    results = fill_mask(sent)
    print(f"\nInput: {sent}")
    print("Top predictions:")
    for r in results[:3]:
        print(f"  [{r['score']:.4f}] {r['sequence']}")

---
# 8. Fine-Tuning for Downstream Tasks

## 8.1 What is Fine-Tuning?

Fine-tuning takes a **pre-trained BERT model** and trains it further on a **specific task** (e.g., sentiment classification, NER, QA).

**Two-step Framework:**
1. **Pre-training**: BERT learns general language representations (MLM + NSP) on massive data
2. **Fine-tuning**: A task-specific head is added; only that head (and sometimes a few BERT layers) is trained

## 8.2 What Happens During Fine-Tuning?

- Pre-trained BERT weights are initialized
- A **task-specific head** is added on top
- The head is trained on labeled task data
- Pre-trained BERT weights are kept **fixed** (or optionally unfrozen)

**Time**: Pre-training took 4 days on multiple TPUs. Fine-tuning takes **a few hours on a GPU**.

## 8.3 Task-Specific Heads

| Task | Head | Input to Head |
|---|---|---|
| **Text Classification** | Linear layer | `[CLS]` token output |
| **NER / Token Classification** | Linear layer per token | Each token's output |
| **Question Answering** | Two linear layers (start + end) | Each token's output |

## 8.4 Prerequisites for Fine-Tuning

```python
pip install transformers
```

You need:
- A pre-trained BERT model (`bert-base-uncased` or `bert-large-uncased`)
- Labeled training data (input sequences + labels)
- A validation set
- A classification head (Linear + optional Dropout)
- Optimizer: **Adam** | Loss: **Cross-Entropy**
- Hyperparameters: learning rate, batch size, epochs

In [ ]:
# Fine-tuning BERT for Text Classification
import torch
import torch.nn as nn
from transformers import BertTokenizer, BertModel

class BertClassifier(nn.Module):
    def __init__(self, num_classes=2, dropout=0.3):
        super().__init__()
        self.bert = BertModel.from_pretrained('bert-base-uncased')
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(768, num_classes)  # 768 = BERT-base hidden size

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        cls_output = outputs.last_hidden_state[:, 0, :]  # [CLS] token
        cls_output = self.dropout(cls_output)
        logits = self.classifier(cls_output)
        return logits

tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertClassifier(num_classes=2)

# Sample texts (Positive / Negative)
texts = [
    "This movie was absolutely fantastic!",
    "Terrible experience, would not recommend.",
    "I loved every moment of it.",
    "The worst product I have ever bought.",
]
labels = torch.tensor([1, 0, 1, 0])  # 1=positive, 0=negative

# Tokenize
encoding = tokenizer(texts, padding=True, truncation=True,
                     max_length=64, return_tensors='pt')

# Forward pass
model.eval()
with torch.no_grad():
    logits = model(encoding['input_ids'], encoding['attention_mask'])
    predictions = torch.argmax(logits, dim=1)

print("=== BERT Text Classification (Fine-Tuning Demo) ===")
print(f"\nModel parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Trainable classifier params: {sum(p.numel() for p in model.classifier.parameters()):,}")
print("\nPredictions (before fine-tuning — random weights in head):")
for text, pred, label in zip(texts, predictions, labels):
    p = 'Positive' if pred == 1 else 'Negative'
    l = 'Positive' if label == 1 else 'Negative'
    print(f"  [{l}] → Predicted: {p} | {text[:50]}...")

In [ ]:
# Fine-tuning for Question Answering — using Hugging Face pipeline
from transformers import pipeline

qa_pipeline = pipeline('question-answering', model='deepset/bert-base-cased-squad2')

context = """
BERT is a transformer model pre-trained by Google on masked language modeling and 
next sentence prediction. RoBERTa improved on BERT by using dynamic masking, 
removing NSP, and training on more data. Both models use transformer encoders 
and can be fine-tuned for tasks like text classification, named entity recognition, 
and question answering.
"""

questions = [
    "Who pre-trained BERT?",
    "What does RoBERTa improve over BERT?",
    "What tasks can BERT be fine-tuned for?",
]

print("=== Fine-Tuned BERT for Question Answering ===")
for q in questions:
    result = qa_pipeline(question=q, context=context)
    print(f"\nQ: {q}")
    print(f"A: {result['answer']} (confidence: {result['score']:.4f})")

---
# 9. Summary — Unit IV

| Topic | Key Concept | Key Contribution |
|---|---|---|
| **RNN** | Hidden state passed through time | Handles sequential data |
| **LSTM** | Cell state + 3 gates | Solves vanishing gradient |
| **GRU** | 2 gates (update + reset) | Simpler, efficient LSTM variant |
| **Attention** | Focus on relevant parts of sequence | Long-range dependency capture |
| **Transformer** | Self-attention + parallel processing | Replaced RNNs for NLP |
| **Self-Attention** | Q·Kᵀ / √d_k → softmax → weighted V | Context-aware word representations |
| **Multi-Head Attention** | Multiple attention heads concatenated | Multiple relationship perspectives |
| **BERT** | Bidirectional encoder, MLM + NSP | Contextual embeddings |
| **RoBERTa** | Dynamic masking, no NSP, more data | Better BERT pretraining |
| **Fine-Tuning** | Add task head to pre-trained BERT | Downstream task adaptation |

---
> **Libraries used:** `torch`, `transformers`, `numpy`  
> **Models used:** `bert-base-uncased`, `roberta-base`, `deepset/bert-base-cased-squad2`